In [1]:
import dspy
from pydantic import BaseModel, Field
from typing import List

local_llm = dspy.LM(
    "openai/qwen3:30b", 
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed"
)

dspy.configure(lm=local_llm,  cache=False)
dspy.configure_cache(enable_disk_cache=False)
dspy.configure_cache(enable_memory_cache=False)

dspy.configure(lm=local_llm)

In [2]:
# Definition des Ziel-Datenmodells
class ProductSchema(BaseModel):
    name: str = Field(..., description="Der genaue Name des Produkts.")
    price: float = Field(..., description="Der Preis des Produkts als Zahl.")
    features: List[str] = Field(..., description="Eine Liste der wichtigsten technischen Merkmale.")

In [3]:
class ProductExtraction(dspy.Signature):
    """
    Analysiert eine Produktbeschreibung und extrahiert strukturierte Daten gemäß dem definierten Schema.
    """
    description: str = dspy.InputField(desc="Der unstrukturierte Werbetext des Produkts.")
    product_data: ProductSchema = dspy.OutputField(desc="Die extrahierten Produktinformationen als Objekt auf deutsch.")

In [4]:
# Erstellen des Predictors mit der definierten Signatur
extractor = dspy.Predict(ProductExtraction)

In [5]:
# Unstrukturierter Eingabetext (Santa Cruz Skateboard)
raw_text = """
Hol dir das Santa Cruz Classic Dot 80s Cruiser Skateboard für dein nächstes Abenteuer. 
Dieses Board kostet aktuell 149,95 Euro und bietet echtes Retro-Feeling. 
Es besteht aus 7-lagigem nordamerikanischem Ahorn und ist mit weichen 
60mm Slime Balls Rollen ausgestattet, die perfekt für rauen Asphalt sind. 
Zudem verfügt es über hochwertige Krux Achsen und das ikonische Logo-Design auf der Unterseite.
"""

# Ausführung der Extraktion
response = extractor(description=raw_text)

# Zugriff auf das extrahierte Pydantic-Objekt
extracted_data = response.product_data

In [6]:
# Ausgabe als JSON-String
print(extracted_data.model_dump_json(indent=2))

{
  "name": "Santa Cruz Classic Dot 80s Cruiser Skateboard",
  "price": 149.95,
  "features": [
    "7-lagiges nordamerikanisches Ahorn",
    "60mm Slime Balls Rollen",
    "Krux Achsen"
  ]
}


In [7]:
local_llm.inspect_history(n=5)





[2025-12-01T07:14:15.478067]

System message:

Your input fields are:
1. `description` (str): Der unstrukturierte Werbetext des Produkts.
Your output fields are:
1. `product_data` (ProductSchema): Die extrahierten Produktinformationen als Objekt auf deutsch.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## description ## ]]
{description}

[[ ## product_data ## ]]
{product_data}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "properties": {"features": {"type": "array", "description": "Eine Liste der wichtigsten technischen Merkmale.", "items": {"type": "string"}, "title": "Features"}, "name": {"type": "string", "description": "Der genaue Name des Produkts.", "title": "Name"}, "price": {"type": "number", "description": "Der Preis des Produkts als Zahl.", "title": "Price"}}, "required": ["name", "price", "features"], "title": "ProductSchema"}

[[ ## completed ## ]]
In adhering to this struc